# OSFI Regulation Assistant: a RAG project
## Part 3: Generate grounded answers with citations

Part 2 built search. Part 3 adds the **generation** half of RAG: an LLM reads the retrieved passages and writes an answer, citing which passage each statement came from.

**What this notebook covers:**
1. Running an open-source LLM **locally in Colab with Ollama**
2. **Prompt design** for grounded answers: use only the passages, cite them, and refuse when the answer isn't there
3. **Guardrails**: a relevance threshold that stops off-topic questions before they reach the LLM, and automatic **citation checking**
4. A **comparison with the LLM alone**, without retrieval, to show why RAG matters
5. Testing on questions the documents **can't** answer

**Runtime: a T4 GPU is required** (*Runtime → Change runtime type → T4 GPU*). The LLM is too slow on a CPU.

## Step 1: Install Ollama and start the server

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama server is running"); break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start. Rerun this cell.")

## Step 2: Download the LLM

**`qwen2.5:7b`** is a 7-billion-parameter open-source model that follows instructions well, including citation formats and "refuse if not found" rules, and it fits comfortably on a free T4 GPU. The download is about 4.7 GB and takes a few minutes. Colab deletes it when the session ends, so it downloads again each session.

To compare models later, change `LLM_MODEL` (for example to `llama3.1:8b`) and rerun the notebook. Comparing models is exactly the kind of evaluation you did at Bow Valley College.

In [ ]:
LLM_MODEL = "qwen2.5:7b"
!ollama pull {LLM_MODEL} 2>&1 | tail -1

## Step 3: Load the search index from Part 2

In [ ]:
!pip -q install sentence-transformers rank_bm25 faiss-cpu nltk

import os, sys, json
import pandas as pd
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/osfi_rag_project"
except Exception as e:
    print("Drive not available, using a local folder:", e)
    PROJECT_DIR = "osfi_rag_project"

INDEX_DIR = os.path.join(PROJECT_DIR, "index")
CODE_DIR = os.path.join(PROJECT_DIR, "osfi_rag")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from osfi_rag.retriever import HybridRetriever

config = json.load(open(os.path.join(INDEX_DIR, "config.json")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
retriever = HybridRetriever.load(INDEX_DIR, SentenceTransformer(config["embed_model"], device=DEVICE),
                                 CrossEncoder(config["rerank_model"], device=DEVICE))
print(f"Loaded {len(retriever.chunks)} chunks | running on {DEVICE}")

## Step 4: Save the answer-generation code as a module

**`generator.py`** contains the prompt and the `RAGAssistant` class. Read the **system prompt** near the top first. It's the most important design decision in this notebook:

- **"Use only facts stated in the passages"** reduces *hallucination*, meaning the model inventing plausible-sounding facts.
- **Numbered citations after every sentence** let a reader check each claim, and let our code check them too.
- **An exact refusal sentence** makes "I don't know" easy to detect automatically.
- **Temperature 0** makes answers repeatable, which matters for evaluation in Part 4.

After the LLM answers, the code **validates the citations**: every `[n]` must point to a passage that was actually provided, and an answer with no citations is flagged as possibly ungrounded.

In [ ]:
%%writefile {CODE_DIR}/generator.py
"""Grounded answer generation: retrieve OSFI passages, then ask an LLM to answer
using only those passages, with numbered citations that are checked afterwards.

Version 2: partial answers get their own status, so a hedged answer that cites
passages is no longer counted as a refusal.
Version 3: partial answers are detected with or without the colon, and the prompt
discourages adding "not covered" notes by habit."""
from __future__ import annotations

import re
import time
from dataclasses import dataclass, field

import requests

NOT_FOUND = "I can't find this in the OSFI documents provided."
PARTIAL_MARKER = "The passages do not cover:"

SYSTEM_PROMPT = f"""You are an assistant that answers questions about Canadian financial regulation
using ONLY the numbered passages from OSFI guidelines provided by the user.

Rules:
1. Use only facts stated in the passages. Do not use outside knowledge.
2. After every sentence that uses a passage, cite it with its number in square brackets, like [1] or [2][3].
3. If NONE of the passages helps answer the question, reply with exactly this sentence and nothing else:
   {NOT_FOUND}
4. If the question asks for something specific that the passages do not contain, answer the rest with
   citations, then end with one sentence starting "{PARTIAL_MARKER}" naming that specific thing.
   Do NOT add this sentence just because more detail could exist: if the passages answer what was asked,
   simply answer. Never use the sentence from rule 3 in an answer that also contains information.
5. Each passage shows its document and section heading. A passage under a heading such as
   "Key Terms > Model" is the definition of that term, even if the term itself is not repeated in the text.
6. Be concise: at most 5 sentences, in plain language."""


class OllamaClient:
    """Minimal client for a local Ollama server's chat endpoint."""

    def __init__(self, model: str, host: str = "http://localhost:11434", temperature: float = 0.0,
                 num_ctx: int = 8192, timeout: int = 300):
        self.model, self.host, self.timeout = model, host, timeout
        self.options = {"temperature": temperature, "num_ctx": num_ctx}

    def chat(self, messages: list[dict], temperature: float | None = None) -> str:
        options = dict(self.options)
        if temperature is not None:
            options["temperature"] = temperature
        r = requests.post(f"{self.host}/api/chat", timeout=self.timeout,
                          json={"model": self.model, "messages": messages, "stream": False,
                                "options": options})
        r.raise_for_status()
        return r.json()["message"]["content"].strip()


@dataclass
class Answer:
    question: str
    answer: str
    status: str                      # "answered", "partial" or "refused"
    citations: list[int]
    invalid_citations: list[int]
    sources: list[dict]
    top_relevance: float | None
    seconds: float
    warnings: list[str] = field(default_factory=list)

    @property
    def refused(self) -> bool:
        return self.status == "refused"


class RAGAssistant:
    """Retrieve passages, build a grounded prompt, call the LLM, and validate the citations."""

    def __init__(self, retriever, llm, k: int = 5, min_relevance: float | None = None):
        self.retriever, self.llm, self.k = retriever, llm, k
        self.min_relevance = min_relevance   # optional guardrail on the re-ranker score

    @staticmethod
    def format_passages(results) -> str:
        return "\n\n".join(f"[{r.rank}] ({r.doc_id} > {r.section.replace(' › ', ' > ')})\n{r.text}"
                           for r in results)

    def answer(self, question: str) -> Answer:
        t0 = time.time()
        results = self.retriever.search(question, k=self.k, method="hybrid_rerank")
        top = results[0].score if results else None

        # Guardrail 1: if nothing relevant was retrieved, don't ask the LLM at all
        if not results or (self.min_relevance is not None and top < self.min_relevance):
            return Answer(question, NOT_FOUND, "refused", [], [], [], top, round(time.time() - t0, 2),
                          ["No passage passed the relevance threshold; the LLM was not called."])

        user_msg = f"Passages:\n\n{self.format_passages(results)}\n\nQuestion: {question}"
        text = self.llm.chat([{"role": "system", "content": SYSTEM_PROMPT},
                              {"role": "user", "content": user_msg}])

        # Guardrail 2: check every citation points to a passage that was actually provided
        cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", text)})
        valid = [n for n in cited if 1 <= n <= len(results)]
        invalid = [n for n in cited if n not in valid]
        says_not_found = NOT_FOUND.lower().rstrip(".") in text.lower()
        if says_not_found and not valid:
            status = "refused"
        elif says_not_found or re.search(r"passages do not (cover|contain|include|address)", text, re.I):
            status = "partial"
        else:
            status = "answered"

        warnings = []
        if invalid:
            warnings.append(f"Answer cites passages that were not provided: {invalid}")
        if status != "refused" and not valid:
            warnings.append("Answer has no citations, so it may not be grounded in the documents.")

        sources = [{"n": r.rank, "doc_id": r.doc_id, "section": r.section, "url": r.url,
                    "chunk_id": r.chunk_id, "relevance": r.score, "cited": r.rank in valid}
                   for r in results]
        return Answer(question, text, status, valid, invalid, sources, top,
                      round(time.time() - t0, 2), warnings)

In [ ]:
import importlib, osfi_rag.generator as gen_module
importlib.reload(gen_module)
from osfi_rag.generator import OllamaClient, RAGAssistant, NOT_FOUND, SYSTEM_PROMPT

llm = OllamaClient(LLM_MODEL)
assistant = RAGAssistant(retriever, llm, k=5)

def show(a):
    print("Q:", a.question)
    print(f"\n{a.answer}\n")
    for s in a.sources:
        mark = "✓ cited" if s["cited"] else ""
        print(f"  [{s['n']}] {s['doc_id']} › {s['section'][:70]}  (relevance {s['relevance']:.2f}) {mark}")
    for w in a.warnings:
        print("  ⚠", w)
    print(f"  ({a.seconds}s)\n" + "-" * 100)

show(assistant.answer("What is the minimum qualifying rate for uninsured mortgages?"))

## Step 5: Why retrieval matters: the same LLM without the documents

Here the LLM answers from memory alone. Compare with the grounded answers: without retrieval, the model can't cite a source, may describe outdated rules, or may state confident details that no OSFI document says. **Check its claims against the cited answers above.**

In [ ]:
def llm_alone(question):
    return llm.chat([{"role": "user", "content": question + " Answer in at most 4 sentences."}])

for q in ["What is the minimum qualifying rate for uninsured mortgages in Canada?",
          "What does OSFI Guideline E-23 require for model risk management?"]:
    print("Q:", q, "\n\nLLM ALONE:\n", llm_alone(q), "\n", "-" * 100)

## Step 6: Choose a relevance threshold (a guardrail)

Some questions have nothing to do with these documents. Rather than letting the LLM try, we can check the **re-ranker's score for the best passage**: if even the best passage is irrelevant, we refuse immediately. This is faster, cheaper, and safer.

The right threshold depends on the data, so we **measure** it: compare the top scores for questions the documents *can* answer with questions they *can't*.

In [ ]:
IN_SCOPE = [
    "What is the minimum qualifying rate for uninsured mortgages?",
    "What is the maximum loan-to-value ratio for a HELOC?",
    "How should lenders calculate the GDS and TDS ratios?",
    "What should mortgage insurers check when verifying a borrower's income?",
    "How does OSFI define a model in its model risk guideline?",
    "How should a bank work out the actual cost of mortgage insurance it charges borrowers?",
    "Do the B-20 rules apply to combined loan plans and other innovative lending products?",
    "Why is high household debt a risk to Canada's financial system?",
]
OUT_OF_SCOPE = [
    "What is the capital gains tax rate in Canada?",
    "How do I open a TFSA account?",
    "What is the best recipe for banana bread?",
    "Who is the current Governor of the Bank of Canada?",
    "What are the rules for cryptocurrency trading platforms?",
]

def top_score(q):
    hits = retriever.search(q, k=1, method="hybrid_rerank")
    return hits[0].score if hits else float("-inf")

scores = pd.DataFrame([{"question": q, "type": t, "top_relevance": round(top_score(q), 2)}
                       for t, qs in [("in scope", IN_SCOPE), ("out of scope", OUT_OF_SCOPE)] for q in qs])
display(scores.sort_values("top_relevance", ascending=False))

lowest_in = scores.loc[scores["type"] == "in scope", "top_relevance"].min()
highest_out = scores.loc[scores["type"] == "out of scope", "top_relevance"].max()
if lowest_in > highest_out:
    MIN_RELEVANCE = round((lowest_in + highest_out) / 2, 2)
    print(f"The two groups separate cleanly. Threshold set halfway between them: {MIN_RELEVANCE}")
else:
    MIN_RELEVANCE = None
    print("The groups overlap, so no threshold is set. The prompt's refusal rule is the only safeguard. "
          "Note which questions overlap: they're worth discussing in your README.")

## Step 7: Ask questions

The assistant now uses the threshold from Step 6. The out-of-scope questions should be refused, either by the threshold before the LLM is called, or by the LLM itself following the refusal rule.

In [ ]:
assistant = RAGAssistant(retriever, llm, k=5, min_relevance=MIN_RELEVANCE)
answers = []
for q in IN_SCOPE + OUT_OF_SCOPE:
    a = assistant.answer(q)
    answers.append(a)
    show(a)

## Step 8: Summary and save

A quick overview of how the assistant behaved. **These checks are automatic**: they confirm the answers are *formatted* correctly (citations valid, refusals detected), but not that every statement is *true*. That needs the evaluation in Part 4.

In [ ]:
summary = pd.DataFrame([{
    "question": a.question[:60],
    "type": "in scope" if a.question in IN_SCOPE else "out of scope",
    "refused": a.refused,
    "citations": len(a.citations),
    "invalid_citations": len(a.invalid_citations),
    "warnings": len(a.warnings),
    "seconds": a.seconds,
} for a in answers])
display(summary)

ins, outs = summary[summary["type"] == "in scope"], summary[summary["type"] == "out of scope"]
print(f"In scope: {(~ins['refused']).sum()}/{len(ins)} answered, "
      f"{(ins['citations'] > 0).sum()}/{len(ins)} with at least one valid citation")
print(f"Out of scope: {outs['refused'].sum()}/{len(outs)} correctly refused")
print(f"Invalid citations in total: {summary['invalid_citations'].sum()} | "
      f"median time per answer: {summary['seconds'].median():.1f}s")

pd.DataFrame([{**{k: v for k, v in vars(a).items() if k != "sources"},
               "sources": json.dumps(a.sources)} for a in answers]) \
  .to_csv(os.path.join(RESULTS_DIR, f"part3_answers_{LLM_MODEL.replace(':', '_')}.csv"), index=False)
json.dump({"llm_model": LLM_MODEL, "min_relevance": MIN_RELEVANCE, "k": 5},
          open(os.path.join(INDEX_DIR, "generation_config.json"), "w"), indent=2)
print("Saved to", RESULTS_DIR)

## Checkpoint

**Send me:**
1. **Step 4:** the first answer with its sources
2. **Step 5:** the two "LLM alone" answers
3. **Step 6:** the relevance-score table and the threshold message
4. **Step 8:** the summary table and the three summary lines
5. **Any answer that looks wrong to you** from Step 7. Mistakes are useful: they'll shape the Part 4 evaluation and make a great "hardest part" story.